<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/CapstoneLTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
import pandas as pd

# Raw GitHub file URL
url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE_01_03_2026%20-%20Lite.xlsx"
#url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE%20Master%20Raw%20Data.xlsx"

# Load Excel file
df = pd.read_excel(url)

# Show first few rows
df.head()

,Site_ID,Cell_ID,Sector_ID,trigger_ID,datetime,traffic_load_mbps,Total Power
0,101,10111,11,1,2026-03-01 00:00,11.90,17.28
1,101,10111,11,2,2026-03-01 00:15,12.03,17.28
2,101,10111,11,3,2026-03-01 00:30,12.08,17.28
3,101,10111,11,4,2026-03-01 00:45,11.96,17.28
4,101,10111,11,5,2026-03-01 01:00,12.48,17.29


In [91]:
url1 = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Site%20information.xlsx"

# Load into dataframe
info_4g = pd.read_excel(url1)
info_4g.head()


,#,Site_ID,Site Name,lbbp_boards,BBU 3900,BBU 3910,rru_count
0,1,101,AIRPORT_MAHE,2,1,1,12
1,2,102,AIRPORT_PRASLIN,1,1,0,4
2,3,103,ANSE_AUX_PIN,2,1,1,12
3,4,104,ANSE_CIMITIERRE,1,1,0,4
4,5,105,ANSE_FAURE,1,1,1,10


In [92]:
url2 = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/LTE%20Traffic%20Bands.xlsx"

# Load into dataframe
ltetrafficbands = pd.read_excel(url2)
ltetrafficbands.head()

,Lower,Upper,UpperValue,lbbp_power,rru_power
0,0,15,15,3,10
1,15,30,30,6,20
2,30,45,45,9,30
3,45,60,60,12,40
4,60,75,75,15,50


In [ ]:
df["BBU3900"] = 0
df["BBU3910"] = 0
df["rru_count"] = 0
df["lbbp_boards"] = 0
df["BBU3900BP"] = 0
df["BBU3910BP"] = 0
df["lbbp_bp"] = 0
df["ltetrfband"] = 0
df["ltetrfbandmax"] = 0
df["ltetrfbandextrapower"] = 0
df["calctotalpower"] = 0
df["rru_base_power"] = 0

df.head()

In [ ]:
# Merge BBU columns from info_4g using Site_ID
df = df.merge(
    info_4g[["Site_ID", "BBU 3900", "BBU 3910"]],
    on="Site_ID",
    how="left"
)

# Fill df columns
df["BBU3900"] = df["BBU 3900"].fillna(0).astype(int)
df["BBU3910"] = df["BBU 3910"].fillna(0).astype(int)

# Remove temporary columns
df.drop(columns=["BBU 3900", "BBU 3910"], inplace=True)

# Preview
df[["Site_ID", "BBU3900", "BBU3910"]].head()

In [ ]:
df = df.merge(
    info_4g[["Site_ID", "rru_count", "lbbp_boards"]],
    on="Site_ID",
    how="left",
    suffixes=("", "_from_4g")
)

# Fill values
df["rru_count"] = df["rru_count_from_4g"].fillna(0).astype(int)
df["lbbp_boards"] = df["lbbp_boards_from_4g"].fillna(0).astype(int)

# Clean up
df.drop(columns=["rru_count_from_4g", "lbbp_boards_from_4g"], inplace=True)

df.head()

In [ ]:
import numpy as np

df["BBU3900BP"] = np.where(
    df["rru_count"] == 0,
    0,
    (55 * df["BBU3900"] / df["rru_count"]).round(2)
)
df["BBU3910BP"] = np.where(
    df["rru_count"] == 0,
    0,
    (65 * df["BBU3910"] / df["rru_count"]).round(2)
)
df.head()

In [ ]:
import numpy as np

df["lbbp_bp"] = np.where(
    df["rru_count"] == 0,
    0,
    (42.5 * df["lbbp_boards"] / df["rru_count"]).round(2)
)
df[["lbbp_boards", "rru_count", "lbbp_bp"]].head()

In [ ]:
def get_lbbp_power(load):
    match = ltetrafficbands[
        (ltetrafficbands["Lower"] <= load) &
        (ltetrafficbands["Upper"] > load)
    ]

    if not match.empty:
        return match.iloc[0]["lbbp_power"]
    return 0  # default if no match

df["ltetrfband"] = df["traffic_load_mbps"].apply(get_lbbp_power)
df[["traffic_load_mbps", "ltetrfband"]].head()

In [ ]:
df.iloc[10:20]

In [ ]:
def get_upper_band(load):
    match = ltetrafficbands[
        (ltetrafficbands["Lower"] <= load) &
        (ltetrafficbands["Upper"] > load)
    ]

    if not match.empty:
        return match.iloc[0]["UpperValue"]
    return 0

df["ltetrfbandmax"] = df["traffic_load_mbps"].apply(get_upper_band)
df.iloc[10:20]

In [ ]:
import numpy as np

df["ltetrfbandextrapower"] = np.where(
    (df["ltetrfbandmax"] == 0) | (df["rru_count"] == 0),
    0,
    (
        (df["traffic_load_mbps"] * df["ltetrfband"]) /
        (df["ltetrfbandmax"] * df["rru_count"])
    ).round(2)
)

In [ ]:
df[[
    "traffic_load_mbps",
    "ltetrfband",
    "ltetrfbandmax",
    "rru_count",
    "ltetrfbandextrapower"
]].head()

In [ ]:
df.iloc[1:2]

In [ ]:
df["calctotalpower"] = (
    df["BBU3900BP"] +
    df["BBU3910BP"] +
    df["lbbp_bp"] +
    df["ltetrfbandextrapower"]
).round(2)

In [ ]:
df.head()

In [ ]:
#df.to_excel("full data.xlsx", index=False)

Maximum power of a Cell

In [ ]:
lte_max_cell = (
    df.groupby(["Site_ID", "Cell_ID"], as_index=False)["calctotalpower"]
    .max()
    .rename(columns={"calctotalpower": "max_cell_power"})
)

lte_max_cell.head()

In [ ]:
lte_max_cell.to_excel("lte_max_cell.xlsx", index=False)

Add Maximum power of all cells per site

In [ ]:
lte_max_power_site = (
    lte_max_cell.groupby("Site_ID", as_index=False)["max_cell_power"]
    .sum()
    .rename(columns={"max_cell_power": "max_power"})
)

# Round to 2 decimal points
lte_max_power_site["max_power"] = (
    lte_max_power_site["max_power"].round(2)
)

lte_max_power_site.head()

In [ ]:
lte_max_power_site.to_excel("lte_max_power_site.xlsx", index=False)